# Movie Review Polarity — Stage 1

This notebook builds a binary sentiment classifier for the released Pang & Lee movie-review data.

```text
0 = negative
1 = positive
```

The main challenge is not making the model as large as possible. The released training set is very small and has many more positive reviews than negative reviews, so the pipeline is designed around generalization and class balance.


## Notebook map

I organized the experiment around a few checkpoints instead of a standard "load / train / test" layout:

**A. Inspect the sample** → **B. Build two text views** → **C. Calibrate on training folds** → **D. Freeze the model** → **E. Evaluate** → **F. Verify the checkpoint**

`public_test.csv` is kept out of model fitting and threshold selection.


In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

seed_tag = 42
train_file = Path("train.csv")
public_file = Path("public_test.csv")
vault = Path("model_checkpoint")
vault.mkdir(exist_ok=True)


# A — Inspect the sample

Before choosing a classifier, I checked the released class counts because ordinary accuracy can be misleading when one class dominates the training data.


In [ ]:
movie_bank = pd.read_csv(train_file)
public_bank = pd.read_csv(public_file)

print("training rows:", len(movie_bank))
print("public rows:", len(public_bank))

print("\ntraining labels")
print(movie_bank["label"].value_counts().sort_index())

print("\npublic labels")
print(public_bank["label"].value_counts().sort_index())

assert set(movie_bank["label"].unique()) <= {0, 1}
assert set(public_bank["label"].unique()) <= {0, 1}


### What the split means for training

The training set contains **180 positive** reviews and only **60 negative** reviews.

A classifier could therefore lean toward label `1` and still appear fairly accurate. I use balanced class weights, stratified folds, and balanced accuracy during calibration so that performance on the smaller negative class matters during model selection.


In [ ]:
review_ink = movie_bank["text"].fillna("").astype(str)
mood_flag = movie_bank["label"].astype(int).to_numpy()

print("positive share:", mood_flag.mean())
print("first review character count:", len(review_ink.iloc[0]))


# B — Two views of the same review

I use two TF-IDF representations in parallel.

### Word lane
Word unigrams and bigrams capture direct vocabulary and short sentiment phrases.

### Character lane
Character 3–5 grams capture fragments and surface patterns. This gives the model some ability to react to words or variants that were not seen as exact vocabulary items during training.

Both branches use a class-weighted linear SVM. This is intentionally simpler than training a large neural model from scratch because there are only 240 training reviews.


In [ ]:
word_lane = Pipeline([
    ("ink_map", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.98,
        sublinear_tf=True,
        max_features=50000,
    )),
    ("margin_machine", LinearSVC(
        C=0.5,
        class_weight="balanced",
        random_state=seed_tag,
    )),
])

char_lane = Pipeline([
    ("fragment_map", TfidfVectorizer(
        analyzer="char_wb",
        lowercase=True,
        ngram_range=(3, 5),
        min_df=2,
        sublinear_tf=True,
        max_features=60000,
    )),
    ("margin_machine", LinearSVC(
        C=0.25,
        class_weight="balanced",
        random_state=seed_tag,
    )),
])


## Training choices

The final classifier uses `LinearSVC`, so a neural-network learning rate and minibatch size are **not applicable**.

The main training controls are:

- word SVM `C = 0.5`
- character SVM `C = 0.25`
- `class_weight="balanced"`
- 5-fold stratified cross-validation
- `random_state = 42`
- balanced accuracy for threshold selection

`C` controls the regularization tradeoff for the linear SVM. Using a regularized sparse model also keeps training practical on a CPU-only laptop.


# C — Calibrate using training folds only

The ensemble weight and final decision threshold are chosen from **out-of-fold scores on `train.csv`**.

Each training review receives a score from a fold model that did not train on that review. I then search over a small set of word/character mixing weights and thresholds and choose the combination with the best balanced accuracy.

The public-test labels are not involved in this step.


In [ ]:
fold_clock = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=seed_tag,
)

word_echo = cross_val_predict(
    word_lane,
    review_ink,
    mood_flag,
    cv=fold_clock,
    method="decision_function",
    n_jobs=1,
)

char_echo = cross_val_predict(
    char_lane,
    review_ink,
    mood_flag,
    cv=fold_clock,
    method="decision_function",
    n_jobs=1,
)

print(word_echo.shape, char_echo.shape)


In [ ]:
best_combo = None

for word_mix in np.linspace(0.0, 1.0, 21):
    fold_signal = word_mix * word_echo + (1.0 - word_mix) * char_echo

    trial_cuts = np.linspace(
        fold_signal.min(),
        fold_signal.max(),
        401,
    )

    for cut_line in trial_cuts:
        fold_guess = (fold_signal >= cut_line).astype(int)

        bacc = balanced_accuracy_score(mood_flag, fold_guess)
        racc = accuracy_score(mood_flag, fold_guess)

        candidate = (bacc, racc, float(word_mix), float(cut_line))

        if best_combo is None or candidate[:2] > best_combo[:2]:
            best_combo = candidate

fold_bacc, fold_acc, word_mix, cut_line = best_combo

print(f"OOF balanced accuracy: {fold_bacc:.4f}")
print(f"OOF raw accuracy:      {fold_acc:.4f}")
print(f"word weight:           {word_mix:.2f}")
print(f"character weight:      {1.0 - word_mix:.2f}")
print(f"decision threshold:    {cut_line:.6f}")


### Why move the threshold?

Balanced class weights help the SVM during fitting, but the default score cutoff does not have to be the best boundary for a 3:1 training split.

Choosing the cutoff from out-of-fold training predictions lets the minority negative class influence the final decision rule without using the public test set for tuning.


# D — Freeze the Stage 1 model

Once calibration is complete, both branches are fitted on all 240 training reviews. The fitted vectorizers, fitted SVMs, ensemble weight, threshold, label mapping, and random seed are saved together so the classifier can later be loaded without fitting again.


In [ ]:
word_lane.fit(review_ink, mood_flag)
char_lane.fit(review_ink, mood_flag)

stage1_bundle = {
    "word_model": word_lane,
    "char_model": char_lane,
    "alpha": word_mix,
    "threshold": cut_line,
    "label_mapping": {0: "negative", 1: "positive"},
    "random_state": seed_tag,
}

joblib.dump(
    stage1_bundle,
    vault / "sentiment_ensemble.joblib",
)

with open(vault / "config.json", "w", encoding="utf-8") as fp:
    json.dump(
        {
            "model": "Word + character TF-IDF LinearSVC ensemble",
            "alpha_word": word_mix,
            "alpha_char": 1.0 - word_mix,
            "decision_threshold": cut_line,
            "random_state": seed_tag,
        },
        fp,
        indent=2,
    )

print("checkpoint saved")


# E — Public-test evaluation

Only after training and calibration are complete do I use `public_test.csv` for evaluation.


In [ ]:
public_ink = public_bank["text"].fillna("").astype(str)
public_truth = public_bank["label"].astype(int).to_numpy()

word_vote = word_lane.decision_function(public_ink)
char_vote = char_lane.decision_function(public_ink)

blend_vote = (
    word_mix * word_vote
    + (1.0 - word_mix) * char_vote
)

public_guess = (blend_vote >= cut_line).astype(int)

score_acc = accuracy_score(public_truth, public_guess)
score_bacc = balanced_accuracy_score(public_truth, public_guess)
score_grid = confusion_matrix(public_truth, public_guess)

print(f"accuracy:          {score_acc:.4f}")
print(f"balanced accuracy: {score_bacc:.4f}")
print()
print(classification_report(
    public_truth,
    public_guess,
    target_names=["negative", "positive"],
    digits=4,
))
print(score_grid)


In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=score_grid,
    display_labels=["negative", "positive"],
).plot(values_format="d")

plt.title("Public Test — Confusion Matrix")
plt.show()


## Public result

The submitted model reached **77.0% accuracy** on the balanced 400-review public test set.

```text
[[163, 37],
 [55, 145]]
```

Rows are true labels and columns are predicted labels.

That means the classifier correctly identified **163 negative reviews** and **145 positive reviews**.


# F — Create the required prediction file


In [ ]:
submission_sheet = pd.DataFrame({
    "id": public_bank["id"],
    "predicted_label": public_guess.astype(int),
})

assert list(submission_sheet.columns) == ["id", "predicted_label"]
assert submission_sheet["predicted_label"].isin([0, 1]).all()
assert len(submission_sheet) == len(public_bank)

submission_sheet.to_csv("public_test_predictions.csv", index=False)
submission_sheet.head()


# G — Check that the saved model reloads

The serialized model should reproduce the same predictions without retraining.


In [ ]:
frozen_copy = joblib.load(vault / "sentiment_ensemble.joblib")

again_word = frozen_copy["word_model"].decision_function(public_ink)
again_char = frozen_copy["char_model"].decision_function(public_ink)

again_signal = (
    frozen_copy["alpha"] * again_word
    + (1.0 - frozen_copy["alpha"]) * again_char
)

again_guess = (
    again_signal >= frozen_copy["threshold"]
).astype(int)

assert np.array_equal(public_guess, again_guess)

print("reload verification passed")


# Design summary

### Small training set
I preferred a regularized sparse-text model over training a high-capacity neural network from scratch. This reduces overfitting risk and keeps the workflow reproducible on a normal laptop.

### Class imbalance
The pipeline uses balanced class weights, stratified folds, and balanced-accuracy threshold selection.

### Evaluation words not seen in training
The character TF-IDF branch can respond to subword fragments and related surface forms instead of relying only on exact whole-word vocabulary matches.

### Checkpointing
The fitted feature extractors and classifiers are stored with the ensemble settings so inference does not require retraining.


# Use of AI

Generative AI was used as a development assistant during Stage 1.

AI assistance included:

- discussing model families that make sense for a very small text dataset;
- comparing a simpler TF-IDF/linear-classifier approach against heavier neural approaches;
- suggesting ways to handle the 3:1 class imbalance;
- helping organize stratified cross-validation and train-only threshold calibration;
- helping structure checkpoint-saving and reload verification code;
- helping format the notebook and repository documentation;
- helping check that the prediction CSV has the exact required column names.

The AI-related requests in this work focused on practical implementation questions such as building a laptop-friendly sentiment classifier, preventing the majority class from dominating predictions, preserving a reloadable Stage 1 checkpoint, and presenting the evaluation clearly.

The final workflow fits the classifier only with the released training examples. The public test set is used for evaluation rather than model fitting.
